In [12]:
!pip install dask

In [7]:
!pip install zarr

In [1]:
# ============================================================
# 03_phase1_QC_v2.ipynb
# Phase 1 QC — Dask/Zarr pipeline, no HVG selection, V(D)J filtering
# ============================================================

# ----------------------------
# Cell 1 — Imports and paths
# ----------------------------
import scanpy as sc
import anndata as ad
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import dask.array as da
import zarr
import scrublet as scr
import scanpy.external as sce
from pathlib import Path

sc.settings.verbosity = 1

PROJECT_DIR = Path(r"C:\Users\annam\Dissertation 2026")
RAW_DIR = PROJECT_DIR / "Data" / "Raw"
PROCESSED_DIR = PROJECT_DIR / "Data" / "Processed"
ZARR_DIR = PROJECT_DIR / "Data" / "Zarr"
FIGURE_DIR = PROJECT_DIR / "figures" / "phase1_qc_v2"
RESULTS_DIR = PROJECT_DIR / "results" / "phase1_qc_v2"

for d in [PROCESSED_DIR, ZARR_DIR, FIGURE_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Paths set up")

Paths set up


In [2]:
# ----------------------------
# Cell 2 — Load raw data
# ----------------------------
adata1 = sc.read_h5ad(RAW_DIR / "GSE114725_raw.h5ad")
adata2 = sc.read_h5ad(RAW_DIR / "GSE176078_raw.h5ad")

print(adata1)
print(adata2)
print("GSE114725 max:", adata1.X.max())
print("GSE176078 max:", adata2.X.max())

AnnData object with n_obs × n_vars = 47016 × 14875
    obs: 'patient', 'tissue', 'replicate', 'cluster'
AnnData object with n_obs × n_vars = 100064 × 29733
    obs: 'Unnamed: 0', 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'percent.mito', 'subtype', 'celltype_subset', 'celltype_minor', 'celltype_major', 'dataset'
GSE114725 max: 1236.0
GSE176078 max: 29831.0


In [3]:
# ----------------------------
# Cell 3 — Convert to Dask arrays (sparse-safe)
# ----------------------------
from scipy.sparse import issparse
import dask.array as da

def to_dask_sparse(adata, chunk_size=2000):
    if issparse(adata.X):
        # Convert sparse matrix to dask WITHOUT densifying
        # Use dask's built-in sparse support
        adata.X = da.from_array(
            adata.X, 
            chunks=(chunk_size, adata.n_vars),
            asarray=False  # keeps it sparse
        )
    else:
        adata.X = da.from_array(adata.X, chunks=(chunk_size, adata.n_vars))
    return adata

adata1 = to_dask_sparse(adata1)
adata2 = to_dask_sparse(adata2)

print("Dask arrays ready")
print(type(adata1.X))
print(adata1.X)
print(adata2.X)

Dask arrays ready
<class 'dask.array.core.Array'>
dask.array<array, shape=(47016, 14875), dtype=float64, chunksize=(2000, 14875), chunktype=scipy.csr_matrix>
dask.array<array, shape=(100064, 29733), dtype=float32, chunksize=(2000, 29733), chunktype=scipy.csc_matrix>


In [4]:
adata1.X = adata1.X.astype('float32')
adata2.X = adata2.X.astype('float32')
print("Both float32")
print(adata1.X.dtype)
print(adata2.X.dtype)

Both float32
float32
float32


In [5]:
# ----------------------------
# Cell 4 — Filter V(D)J genes before anything else
# ----------------------------
vdj_prefixes = (
    "IGHV", "IGLV", "IGKV",
    "TRAV", "TRBV", "TRGV", "TRDV",
    "IGHD", "IGHJ", "IGLJ", "IGKJ"
)

def filter_vdj_genes(adata, dataset_name):
    before = adata.n_vars
    mask = ~adata.var_names.str.startswith(vdj_prefixes)
    adata = adata[:, mask].copy()
    print(f"{dataset_name}: removed {before - adata.n_vars} V(D)J genes, {adata.n_vars} remaining")
    return adata

adata1 = filter_vdj_genes(adata1, "GSE114725")
adata2 = filter_vdj_genes(adata2, "GSE176078")

GSE114725: removed 24 V(D)J genes, 14851 remaining
GSE176078: removed 345 V(D)J genes, 29388 remaining


In [6]:
# ----------------------------
# Cell 5 — QC metrics (chunk by chunk, no full materialisation)
# ----------------------------
from scipy.sparse import issparse, csr_matrix

def calculate_qc_chunked(adata, dataset_name, chunk_size=2000):
    print(f"Calculating QC metrics for {dataset_name}...")
    
    n_cells = adata.n_obs
    n_genes_by_counts = np.zeros(n_cells, dtype=np.float32)
    total_counts = np.zeros(n_cells, dtype=np.float32)
    pct_counts_mt = np.zeros(n_cells, dtype=np.float32)
    
    mt_mask = adata.var_names.str.startswith("MT-")
    
    for start in range(0, n_cells, chunk_size):
        end = min(start + chunk_size, n_cells)
        
        # Get chunk as sparse matrix
        chunk = adata.X[start:end]
        if isinstance(chunk, da.Array):
            chunk = chunk.compute()
        if not issparse(chunk):
            chunk = csr_matrix(chunk)
        
        # Calculate metrics for this chunk
        total_counts[start:end] = np.asarray(chunk.sum(axis=1)).flatten()
        n_genes_by_counts[start:end] = np.asarray((chunk > 0).sum(axis=1)).flatten()
        
        # Mitochondrial counts
        chunk_mt = chunk[:, mt_mask]
        mt_counts = np.asarray(chunk_mt.sum(axis=1)).flatten()
        total = total_counts[start:end]
        pct_counts_mt[start:end] = np.where(total > 0, (mt_counts / total) * 100, 0)
        
        if start % 10000 == 0:
            print(f"  Processed {end}/{n_cells} cells...")
    
    adata.obs["n_genes_by_counts"] = n_genes_by_counts
    adata.obs["total_counts"] = total_counts
    adata.obs["pct_counts_mt"] = pct_counts_mt
    
    print(f"  Done. Mean genes per cell: {n_genes_by_counts.mean():.0f}")
    print(f"  Mean total counts: {total_counts.mean():.0f}")
    print(f"  Mean MT%: {pct_counts_mt.mean():.2f}%")
    
    return adata

adata1 = calculate_qc_chunked(adata1, "GSE114725")
adata2 = calculate_qc_chunked(adata2, "GSE176078")

print("\nGSE114725 QC summary:")
print(adata1.obs[["n_genes_by_counts", "total_counts", "pct_counts_mt"]].describe())
print("\nGSE176078 QC summary:")
print(adata2.obs[["n_genes_by_counts", "total_counts", "pct_counts_mt"]].describe())

Calculating QC metrics for GSE114725...
  Processed 2000/47016 cells...
  Processed 12000/47016 cells...
  Processed 22000/47016 cells...
  Processed 32000/47016 cells...
  Processed 42000/47016 cells...
  Done. Mean genes per cell: 611
  Mean total counts: 1515
  Mean MT%: 7.37%
Calculating QC metrics for GSE176078...
  Processed 2000/100064 cells...
  Processed 12000/100064 cells...
  Processed 22000/100064 cells...
  Processed 32000/100064 cells...
  Processed 42000/100064 cells...
  Processed 52000/100064 cells...
  Processed 62000/100064 cells...
  Processed 72000/100064 cells...
  Processed 82000/100064 cells...
  Processed 92000/100064 cells...
  Processed 100064/100064 cells...
  Done. Mean genes per cell: 1776
  Mean total counts: 6982
  Mean MT%: 6.19%

GSE114725 QC summary:
       n_genes_by_counts  total_counts  pct_counts_mt
count       47016.000000  47016.000000   47016.000000
mean          610.935669   1515.447021       7.371400
std           454.622864   1604.428589    

In [7]:
# ----------------------------
# Cell 6 — QC plots before filtering
# ----------------------------
for adata, name in [(adata1, "GSE114725"), (adata2, "GSE176078")]:
    sc.pl.violin(
        adata,
        ["n_genes_by_counts", "total_counts", "pct_counts_mt"],
        jitter=0.4,
        multi_panel=True,
        show=False
    )
    plt.savefig(FIGURE_DIR / f"{name}_before_filtering_violin.png", dpi=300, bbox_inches="tight")
    plt.close()

    sc.pl.scatter(
        adata,
        x="total_counts",
        y="n_genes_by_counts",
        color="pct_counts_mt",
        show=False
    )
    plt.savefig(FIGURE_DIR / f"{name}_counts_vs_genes_scatter.png", dpi=300, bbox_inches="tight")
    plt.close()

print("QC plots saved")

QC plots saved


In [8]:
# ----------------------------
# Cell 7 — Filter low quality cells and genes
# ----------------------------
print("Before filtering:")
print(f"GSE114725: {adata1.n_obs} cells, {adata1.n_vars} genes")
print(f"GSE176078: {adata2.n_obs} cells, {adata2.n_vars} genes")

# GSE114725 — immune enriched dataset
# Threshold justification: mean genes = 611, MT% mean = 7.37%, 75th percentile = 9.28%
# n_genes > 200 removes empty droplets below the clear distribution inflection
# pct_mt < 15 is lenient given the naturally high MT% in immune cells
adata1 = adata1[
    (adata1.obs["n_genes_by_counts"] > 200) &
    (adata1.obs["pct_counts_mt"] < 15)
].copy()

# GSE176078 — full TME dataset
# Threshold justification: mean genes = 1776, 25th percentile = 815
# n_genes > 500 removes low quality cells below the main distribution
# pct_mt < 20 accounts for higher MT% ceiling seen in this dataset (max 34.9%)
adata2 = adata2[
    (adata2.obs["n_genes_by_counts"] > 500) &
    (adata2.obs["pct_counts_mt"] < 20)
].copy()

# Filter lowly expressed genes
sc.pp.filter_genes(adata1, min_cells=3)
sc.pp.filter_genes(adata2, min_cells=3)

print("\nAfter filtering:")
print(f"GSE114725: {adata1.n_obs} cells, {adata1.n_vars} genes")
print(f"GSE176078: {adata2.n_obs} cells, {adata2.n_vars} genes")

Before filtering:
GSE114725: 47016 cells, 14851 genes
GSE176078: 100064 cells, 29388 genes

After filtering:
GSE114725: 44533 cells, 14828 genes
GSE176078: 92236 cells, 27379 genes


In [11]:
# ----------------------------
# Cell 8 — Scrublet doublet detection (per dataset)
# ----------------------------
def run_scrublet_per_sample(adata, dataset_name, sample_key, chunk_size=2000):
    print(f"Running per-sample Scrublet for {dataset_name}")
    
    from scipy.sparse import vstack, csr_matrix
    
    samples = adata.obs[sample_key].unique()
    print(f"  Found {len(samples)} samples")
    
    all_doublet_scores = np.zeros(adata.n_obs, dtype=np.float32)
    all_predicted_doublets = np.zeros(adata.n_obs, dtype=bool)
    
    for sample in samples:
        # Get cell indices for this sample
        sample_mask = adata.obs[sample_key] == sample
        sample_idx = np.where(sample_mask)[0]
        n_cells = len(sample_idx)
        
        print(f"  Sample {sample}: {n_cells} cells")
        
        if n_cells < 50:
            print(f"    Skipping — too few cells")
            continue
        
        # Extract sparse matrix for this sample chunk by chunk
        chunks = []
        for start in range(0, n_cells, chunk_size):
            end = min(start + chunk_size, n_cells)
            global_idx = sample_idx[start:end]
            chunk = adata.X[global_idx].compute()
            if not issparse(chunk):
                chunk = csr_matrix(chunk)
            chunks.append(chunk)
        
        X_sample = vstack(chunks)
        
        # Run Scrublet on this sample
        try:
            scrub = scr.Scrublet(X_sample)
            doublet_scores, predicted_doublets = scrub.scrub_doublets(
                verbose=False
            )
            all_doublet_scores[sample_idx] = doublet_scores
            all_predicted_doublets[sample_idx] = predicted_doublets
            n_doublets = predicted_doublets.sum()
            print(f"    Doublets detected: {n_doublets} ({100*n_doublets/n_cells:.1f}%)")
            
        except Exception as e:
            print(f"    Scrublet failed for {sample}: {e}")
            continue
    
    adata.obs["doublet_score"] = all_doublet_scores
    adata.obs["predicted_doublet"] = all_predicted_doublets
    
    before = adata.n_obs
    adata = adata[~adata.obs["predicted_doublet"]].copy()
    
    print(f"\n  Total cells before: {before}")
    print(f"  Total cells after: {adata.n_obs}")
    print(f"  Total doublets removed: {before - adata.n_obs}")
    
    # Save doublet score violin
    sc.pl.violin(adata, ["doublet_score"], jitter=0.4, show=False)
    plt.savefig(
        FIGURE_DIR / f"{dataset_name}_scrublet_scores.png",
        dpi=300,
        bbox_inches="tight"
    )
    plt.close()
    
    return adata

# GSE114725 — batch by patient
adata1 = run_scrublet_per_sample(adata1, "GSE114725", sample_key="patient")

# GSE176078 — batch by sample identifier
adata2 = run_scrublet_per_sample(adata2, "GSE176078", sample_key="orig.ident")

Running per-sample Scrublet for GSE114725
  Found 8 samples
  Sample BC5: 1758 cells
    Doublets detected: 5 (0.3%)
  Sample BC7: 1858 cells
    Doublets detected: 7 (0.4%)
  Sample BC6: 3477 cells
    Doublets detected: 2 (0.1%)
  Sample BC2: 8129 cells
    Doublets detected: 2 (0.0%)
  Sample BC4: 18989 cells
    Doublets detected: 653 (3.4%)
  Sample BC8: 2550 cells
    Doublets detected: 1 (0.0%)
  Sample BC3: 852 cells
    Doublets detected: 2 (0.2%)
  Sample BC1: 6920 cells
    Doublets detected: 3 (0.0%)

  Total cells before: 44533
  Total cells after: 43858
  Total doublets removed: 675
Running per-sample Scrublet for GSE176078
  Found 26 samples
  Sample CID3586: 5825 cells
    Doublets detected: 1 (0.0%)
  Sample CID3921: 2848 cells
    Doublets detected: 0 (0.0%)
  Sample CID45171: 2411 cells
    Doublets detected: 4 (0.2%)
  Sample CID3838: 2348 cells
    Doublets detected: 1 (0.0%)
  Sample CID4066: 5053 cells
    Doublets detected: 0 (0.0%)
  Sample CID44041: 2066 cells

In [14]:
# Check a small chunk of raw values
import numpy as np
chunk = adata1.X[0:10].compute()
print("Sample values from adata1:")
print(chunk.toarray()[0, :20])

chunk2 = adata2.X[0:10].compute()
print("Sample values from adata2:")
print(chunk2.toarray()[0, :20])

Sample values from adata1:
[0.       0.       0.       0.       0.       0.       0.       0.
 0.       3.658301 0.       0.       0.       0.       0.       0.
 0.       0.       0.       0.      ]
Sample values from adata2:
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]


In [13]:
# ----------------------------
# Cell 9 — Normalisation and save raw counts
# ----------------------------
sc.pp.normalize_total(adata1, target_sum=1e4)
sc.pp.log1p(adata1)

sc.pp.normalize_total(adata2, target_sum=1e4)
sc.pp.log1p(adata2)

# Save raw normalised counts before scaling — critical for CellTypist later
adata1.raw = adata1
adata2.raw = adata2

print("Normalised and raw counts stored")
print("GSE114725 max:", adata1.X[0:100].compute().max())
print("GSE176078 max:", adata2.X[0:100].compute().max())
print("GSE114725 shape:", adata1.shape)
print("GSE176078 shape:", adata2.shape)

Normalised and raw counts stored
GSE114725 max: 4.380037
GSE176078 max: 3.9132452
GSE114725 shape: (43858, 14828)
GSE176078 shape: (92042, 27379)


In [ ]:
#checkpoint save 
adata1.write(PROCESSED_DIR / "GSE114725_phase1_v2_prenormonly.h5ad", compression="gzip")
adata2.write(PROCESSED_DIR / "GSE176078_phase1_v2_prenormonly.h5ad", compression="gzip")

In [ ]:
# ----------------------------
# Cell 10 — PCA and Harmony (no scaling, full gene matrix)
# ----------------------------
import scanpy.external as sce

# PCA directly on log-normalised data — no scaling required
sc.tl.pca(adata1, svd_solver="arpack", n_comps=50)
sc.tl.pca(adata2, svd_solver="arpack", n_comps=50)

print("PCA complete")

# Harmony batch correction
sce.pp.harmony_integrate(adata1, key="patient", basis="X_pca")
sce.pp.harmony_integrate(adata2, key="orig.ident", basis="X_pca")

print("Harmony complete")
print(adata1.obsm.keys())
print(adata2.obsm.keys())

In [ ]:
# ----------------------------
# Cell 11 — Neighbours and UMAP before/after Harmony
# ----------------------------
def run_neighbors_umap(adata, use_rep, random_state=42):
    sc.pp.neighbors(adata, use_rep=use_rep, n_neighbors=15, n_pcs=30)
    sc.tl.umap(adata, random_state=random_state)
    return adata

# Before Harmony — use raw PCA
for adata, name, batch_key in [
    (adata1, "GSE114725", "patient"),
    (adata2, "GSE176078", "orig.ident")
]:
    sc.pp.neighbors(adata, use_rep="X_pca", n_neighbors=15, n_pcs=30)
    sc.tl.umap(adata, random_state=42)

    sc.pl.umap(adata, color=[batch_key], title=f"{name} before Harmony", show=False)
    plt.savefig(FIGURE_DIR / f"{name}_before_harmony.png", dpi=300, bbox_inches="tight")
    plt.close()

# After Harmony
for adata, name, batch_key in [
    (adata1, "GSE114725", "patient"),
    (adata2, "GSE176078", "orig.ident")
]:
    sc.pp.neighbors(adata, use_rep="X_pca_harmony", n_neighbors=15, n_pcs=30)
    sc.tl.umap(adata, random_state=42)

    sc.pl.umap(adata, color=[batch_key], title=f"{name} after Harmony", show=False)
    plt.savefig(FIGURE_DIR / f"{name}_after_harmony.png", dpi=300, bbox_inches="tight")
    plt.close()

print("UMAP plots saved")

In [ ]:
# ----------------------------
# Cell 12 — Save final processed objects
# ----------------------------
adata1.write(
    PROCESSED_DIR / "GSE114725_phase1_v2.h5ad",
    compression="gzip"
)
adata2.write(
    PROCESSED_DIR / "GSE176078_phase1_v2.h5ad",
    compression="gzip"
)

print("Saved:")
print(adata1)
print(adata2)